# 1.0 – EDA: Khám phá tín hiệu EEG (DEAP Dataset)

**Mục tiêu notebook này:**
- Đọc file `.dat` của DEAP và kiểm tra cấu trúc dữ liệu
- Vẽ tín hiệu thô của từng kênh EEG theo thời gian
- Phân tích phân bố nhãn Valence / Arousal
- Quan sát phổ tần số (FFT) cơ bản

**Tài liệu tham khảo:**  
DEAP: A Database for Emotion Analysis using Physiological Signals – Koelstra et al. (2012)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '../data/raw'   # Đường dẫn đến thư mục chứa file .dat
SUBJECT  = 's01.dat'       # Đọc thử subject đầu tiên
FS       = 128             # Hz

print('NumPy version :', np.__version__)

## 1. Đọc & kiểm tra cấu trúc dữ liệu

In [ ]:
path = os.path.join(DATA_DIR, SUBJECT)
with open(path, 'rb') as f:
    subject = pickle.load(f, encoding='latin1')

data   = subject['data']    # (40, 40, 8064)
labels = subject['labels']  # (40, 4) – valence, arousal, dominance, liking

print('data   shape:', data.shape)
print('labels shape:', labels.shape)
print('\nLabel columns: valence | arousal | dominance | liking')
print('Label range  :', labels.min(), '→', labels.max())

## 2. Vẽ tín hiệu thô – 1 trial, 8 kênh EEG đầu tiên

In [ ]:
TRIAL_IDX = 0
N_CH_SHOW = 8
t = np.arange(data.shape[2]) / FS   # trục thời gian (giây)

fig, axes = plt.subplots(N_CH_SHOW, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'Trial {TRIAL_IDX} – 8 kênh EEG đầu tiên', fontsize=14)

for i, ax in enumerate(axes):
    ax.plot(t, data[TRIAL_IDX, i, :], lw=0.6, color=f'C{i}')
    ax.set_ylabel(f'Ch {i+1}', fontsize=8)
    ax.axvline(3, color='red', lw=1, linestyle='--', label='Baseline end' if i == 0 else '')

axes[0].legend(fontsize=8)
axes[-1].set_xlabel('Thời gian (s)')
plt.tight_layout()
plt.show()

## 3. Phân bố nhãn Valence & Arousal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for ax, idx, name in zip(axes, [0, 1], ['Valence', 'Arousal']):
    ax.hist(labels[:, idx], bins=10, color='steelblue', edgecolor='white')
    ax.axvline(5.0, color='red', lw=1.5, linestyle='--', label='Threshold 5.0')
    ax.set_title(name)
    ax.set_xlabel('Score (1–9)')
    ax.set_ylabel('Số trial')
    ax.legend()

plt.suptitle(f'Phân bố nhãn – {SUBJECT}', y=1.02)
plt.tight_layout()
plt.show()

# Tỉ lệ nhị phân
for name, col in [('Valence', 0), ('Arousal', 1)]:
    low  = (labels[:, col] < 5).sum()
    high = (labels[:, col] >= 5).sum()
    print(f'{name}: Low={low}  High={high}  (tổng={low+high})')

## 4. Phổ tần số (FFT) – kênh Fp1, 1 trial

In [ ]:
from scipy.signal import welch

# Lấy kênh 0 (Fp1), trial 0, bỏ 3 giây baseline
signal = data[TRIAL_IDX, 0, 3 * FS:]

freqs, psd = welch(signal, fs=FS, nperseg=FS * 2)

# Các dải tần
BANDS = {'theta (4-8)': (4, 8), 'alpha (8-13)': (8, 13),
         'beta (13-30)': (13, 30), 'gamma (30-45)': (30, 45)}

plt.figure(figsize=(10, 4))
plt.semilogy(freqs, psd, color='steelblue', lw=1.5)

colors = ['#FFA500', '#4CAF50', '#E91E63', '#9C27B0']
for (name, (lo, hi)), color in zip(BANDS.items(), colors):
    mask = (freqs >= lo) & (freqs <= hi)
    plt.fill_between(freqs, psd, where=mask, alpha=0.4, color=color, label=name)

plt.xlabel('Tần số (Hz)')
plt.ylabel('PSD (V²/Hz)')
plt.title('Welch PSD – Kênh Fp1, Trial 0')
plt.legend()
plt.xlim(0, 50)
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()